# Operational KPI Dashboard

A reproducible walkthrough of data quality checks, KPI calculations and reporting outputs.

> **Data note:** This portfolio reconstruction uses a reproducible synthetic dataset because the original training files were not available for publication.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data/processed/operations_clean.csv", parse_dates=["date"])
df.head()

## Data-quality checks

In [ ]:
quality = pd.Series({
    "rows": len(df),
    "duplicate_transaction_ids": df.transaction_id.duplicated().sum(),
    "missing_required_fields": df[["date","units_processed","target_units"]].isna().sum().sum(),
    "negative_units": (df.units_processed < 0).sum(),
})
quality

## KPI summary

In [ ]:
summary = {
    "target_attainment_pct": 100 * df.units_processed.sum() / df.target_units.sum(),
    "defect_rate_pct": 100 * df.defects.sum() / df.units_processed.sum(),
    "on_time_rate_pct": 100 * df.on_time_flag.mean(),
    "gross_margin_gbp": df.gross_margin_gbp.sum(),
}
pd.Series(summary).round(2)

## Monthly trend

In [ ]:
monthly = df.assign(month=df.date.dt.to_period("M").astype(str)).groupby("month").agg(units=("units_processed","sum"), target=("target_units","sum")).reset_index()
monthly["attainment_pct"] = monthly.units / monthly.target * 100
monthly.plot(x="month", y="attainment_pct", marker="o", figsize=(12,5), title="Monthly target attainment")
plt.xticks(rotation=45); plt.tight_layout();